In [0]:

CREATE OR REPLACE FUNCTION cineai.app.get_screening(
    p_screening_id STRING COMMENT 'Exact screening identifier'
)
RETURNS STRING
COMMENT 'Retrieves a screening and calculates temporal eligibility fields. Input: screening ID. Output: JSON with screening details, is_past, minutes_until_start, and cancellation_cutoff_passed. Returns NOT_FOUND when the screening does not exist. Invalid source data or SQL failures raise an error.'
RETURN
SELECT COALESCE(
    MAX(
        to_json(
            named_struct(
                'screening_id', screening_id,
                'movie_id', movie_id,
                'cinema_name', cinema_name,
                'city', city,
                'screening_date', screening_date,
                'start_time', start_time,
                'screening_status', status,
                'price_eur', price_eur,
                'seats_available', seats_available,
                'last_updated_at', last_updated_at,
                'timezone', 'Europe/Madrid',
                'is_past', screening_start_utc < now_utc,
                'minutes_until_start',
                    timestampdiff(
                        MINUTE,
                        now_utc,
                        screening_start_utc
                    ),
                'cancellation_cutoff_passed',
                    now_utc > screening_start_utc - INTERVAL 2 HOURS,
                'checked_at_utc', CAST(now_utc AS STRING)
            )
        )
    ),
    'NOT_FOUND'
)
FROM (
    SELECT
        s.*,
        convert_timezone(
            'Europe/Madrid',
            'UTC',
            to_timestamp_ntz(
                concat(
                    CAST(s.screening_date AS STRING),
                    ' ',
                    substr(CAST(s.start_time AS STRING), 1, 5)
                ),
                'yyyy-MM-dd HH:mm'
            )
        ) AS screening_start_utc,
        convert_timezone(
            current_timezone(),
            'UTC',
            localtimestamp()
        ) AS now_utc
    FROM cineai.raw.screenings s
    WHERE s.screening_id = p_screening_id
) AS screening_data;